In [10]:
import pandas as pd
import numpy as np

In [11]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Sonia_Vihar, Delhi - DPCC.xlsx",skiprows=16)

In [12]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,113.73,190.63,16.85,34.32,31.95,1.02,26.72,NaN,NaN,80.33,0.57,201.28,85.37,994.16,12.76,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,137.74,235.91,32.76,29.74,42.45,1.57,24.75,NaN,NaN,81.13,0.62,246.14,70.80,994.15,12.94,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,166.46,269.92,36.73,37.60,49.86,1.85,18.18,NaN,NaN,85.55,0.43,220.68,69.33,994.26,13.43,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,214.21,339.79,39.04,59.11,63.18,2.38,18.28,NaN,NaN,85.09,0.81,130.96,75.16,994.17,13.98,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,158.88,288.33,25.84,48.45,46.78,1.40,20.48,NaN,NaN,78.80,1.24,164.82,94.29,993.91,14.13,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,365.62,525.17,44.01,48.57,61.61,2.85,65.15,6.35,13.78,68.12,0.53,29.29,88.98,985.61,19.75,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,308.62,468.08,49.88,57.09,70.92,3.02,52.51,10.24,22.52,71.00,0.42,56.31,150.70,986.35,19.63,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,255.96,410.62,28.88,54.48,52.46,2.14,51.10,6.71,15.51,70.67,0.43,93.62,65.22,986.30,19.38,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,278.96,428.88,31.88,49.86,52.44,2.39,53.44,5.97,13.54,69.75,0.42,121.30,54.51,986.73,19.31,0.0,0.0


In [13]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [14]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [15]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [16]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 19)
          From Date           To Date    PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  113.730  190.63  16.85  34.32  31.95   
1  02-01-2025 00:00  03-01-2025 00:00  137.740  235.91  32.76  29.74  42.45   
2  03-01-2025 00:00  04-01-2025 00:00   57.665  269.92  36.73  37.60  49.86   
3  04-01-2025 00:00  05-01-2025 00:00   57.665  339.79  39.04  59.11  63.18   
4  05-01-2025 00:00  06-01-2025 00:00  158.880  288.33  25.84  48.45  46.78   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.02  26.72    3.005     8.57  80.33  0.57  201.28  85.37  994.16  12.76   
1  1.57  24.75    3.005     8.57  81.13  0.62  246.14  70.80  994.15  12.94   
2  1.85  18.18    3.005     8.57  85.55  0.43  220.68  69.33  994.26  13.43   
3  2.38  18.28    3.005     8.57  85.09  0.81  130.96  75.16  994.17  13.98   
4  1.40  20.48    3.005     8.57  78.80  1.24  164.82  94.29  993.91  14.13   

    RF  TOT-RF  
0  0.0    

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [18]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,1.700792,0.366180,0.194448,0.110597,0.126864,-0.594758,-0.430584,0.101457,0.150695,1.151228,-0.867721,0.539261,-0.226852,1.991875,-2.226879,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.476101,0.945295,1.732630,-0.220784,0.829764,0.209173,-0.527208,0.101457,0.150695,1.214055,-0.805025,1.361692,-0.474648,1.989979,-2.199477,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.109606,1.380270,2.116450,0.347918,1.325811,0.618447,-0.849453,0.101457,0.150695,1.561174,-1.043270,0.894927,-0.499648,2.010837,-2.124882,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.109606,2.273882,2.339781,1.904250,2.217490,1.393144,-0.844548,0.101457,0.150695,1.525049,-0.566780,-0.749935,-0.400496,1.993771,-2.041152,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,3.158734,1.615727,1.063603,1.132958,1.119627,-0.039314,-0.736642,0.101457,0.150695,1.031072,-0.027594,-0.129170,-0.075148,1.944469,-2.018317,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,-0.109606,-0.124174,2.820282,1.141640,2.112390,2.080140,1.454327,0.101457,2.871234,0.192331,-0.917878,-2.613879,-0.165456,0.370602,-1.162758,0.0,0.0
315,13-11-2025 00:00,14-11-2025 00:00,-0.109606,-0.124174,-0.275417,1.758095,2.735628,2.328628,0.834362,0.101457,0.150695,0.418509,-1.055810,-2.118514,0.884230,0.510923,-1.181026,0.0,0.0
316,14-11-2025 00:00,15-11-2025 00:00,-0.109606,-0.124174,1.357511,1.569252,1.499863,1.042338,0.765204,0.101457,0.150695,0.392592,-1.043270,-1.434499,-0.569548,0.501442,-1.219085,0.0,0.0
317,15-11-2025 00:00,16-11-2025 00:00,-0.109606,-0.124174,1.647551,1.234977,1.498524,1.407761,0.879976,2.682108,2.745912,0.320341,-1.055810,-0.927034,-0.751696,0.582979,-1.229741,0.0,0.0


In [19]:
df.to_excel('soniavihar2025.xlsx', index=False)